# Performance Measurement · make the numbers you'll defend in your report

This is **lab 02**. Lab 01 gave you the serial baseline — one number, MLUP/s. Lab 02 makes measurement itself the subject. By the end of this lab you know:

1. **How to time correctly** — wall vs CPU vs user vs system, and why one number can lie without the others.
2. **How to read a `perf stat` report** — instructions per cycle, cache miss rate, branch prediction accuracy, and how to translate those into ideas for speedup.
3. **How to draw a real roofline** for Crux by measuring the two ceilings (memory bandwidth, peak FLOPs) yourself, and where lab 01's heat stencil sits on it.
4. **How to measure power** with `perf stat -e power/energy-pkg/` and, on Crux, from PBS job accounting — and add an `energy_j` column to `timings.csv`.
5. **How to run a controlled A/B experiment** — swap one compile flag, re-measure, plot both against the roofline, and answer *why* the number changed.

**Prerequisites.** You have finished lab 01 (you have `heat2D.c` on Crux with a working stencil, and you produced a `timings.csv` with at least one row). If your labCC section on timers (Part 5) is a distant memory, skim it before starting Part 1 here.

> **📚 Where to look when you're stuck**
> 
> - [**`perf` wiki**](https://perfwiki.github.io/main/) — the Linux perf tool. Read > Tutorial + Common Errors first.
> - [**Roofline paper (Williams, Waterman, Patterson 2009)**](https://people.eecs.berkeley.edu/~kubitron/cs252/handouts/papers/RooflineVyNoYellow.pdf) > — 12 pages, the origin.
> - [**STREAM benchmark**](https://www.cs.virginia.edu/stream/) — where the triad > kernel comes from. The reference for memory bandwidth on any x86 machine.
> - [**Crux Running Jobs**](https://docs.alcf.anl.gov/crux/queueing-and-running-jobs/running-jobs/) > — PBS Pro reference (you'll edit `heat2D.pbs` again in Parts 2 and 4).


## How this notebook works · same three surfaces as lab 01

| Where | How it looks | What it can do |
|---|---|---|
| **Hub** | plain Python or `!cmd` | plots, `pandas`, orchestration |
| **Crux login** | `sshRun('cmd')` | edit files, build, `qsub`, `qstat` |
| **Crux compute** | inside a PBS script | the actual measurements |

Every code cell begins with `# [Where]`. Nothing new since lab 00.


In [ ]:
# [Hub] Shared toolkit.
from labHelpers import *


### Set up this lab's identity


In [ ]:
# [Hub] Change HPC_USER to your ALCF username; re-run.
env = setupLab(labName="lab02", host="crux",
               remoteUser=os.environ.get("HPC_USER","CHANGE_ME"),
               project="UIC-CS455-Sp2027", queue="debug",
               scratch=f"/eagle/UIC-CS455-Sp2027/{os.environ.get('HPC_USER','CHANGE_ME')}")
labDir = pathlib.Path(env['labDir'])


### Preflight


In [ ]:
# [Hub] Reachability + a check that lab01 produced the CSV we'll extend.
preflight([
    check("passwordless ssh to Crux", sshReachable()),
    check("scheduler answers", schedulerAnswers()),
    check("lab02 dir on Crux", remoteFileExists(env['HPC_LAB_DIR']),
          hint="next cell creates it if missing; re-run this after mkdir"),
    check("lab01 timings.csv exists on Crux",
          remoteFileExists(env['HPC_LAB_DIR'].replace('lab02','lab01') + '/out/timings.csv'),
          hint="do lab 01 Part 4 first - lab 02 extends lab 01's CSV"),
], infoRows=[('cluster', clusterHost()), ('you', env.get('HPC_USER','?')),
             ('project', env.get('HPC_PROJECT','?')), ('lab dir', env.get('HPC_LAB_DIR','?'))])


In [ ]:
# [Hub -> Crux] Make the lab dir if it wasn't there.
sshRun(f'mkdir -p {env["HPC_LAB_DIR"]}/out', quiet=True)
print('lab02 dir ready')


## Part 1 · Time correctly · wall vs CPU vs user vs system

When a program takes 10 seconds, that could mean:

- **Wall time** (10s) — actual elapsed seconds. What a user sees.
- **User CPU** (9.5s) — seconds the CPU spent running your code.
- **System CPU** (0.3s) — seconds the kernel spent on your behalf (I/O, memory ops).
- **User + system < wall** (9.8s ≪ 10s) — the program spent time *blocked* on I/O or a lock, waiting for something.
- **User + system > wall** — the program used more than one core in parallel. On a 64-core node with perfect parallelism, `user ~ 64 * wall`.

Every HPC report cites **wall time** as the primary number. It's the one users pay for and the one that fills the queue. But wall alone doesn't tell you *why* — for that, you need the other three.

The shell command `time ./program` gives you all four. On Linux, `/usr/bin/time -v` (the GNU one, not the shell builtin) gives you memory too. From C, [`clock_gettime(CLOCK_MONOTONIC, ...)`](https://man7.org/linux/man-pages/man2/clock_gettime.2.html) is the wall-time source of truth (labCC Part 5).


In [ ]:
# [Hub -> Crux] Run heat2D under GNU time on a compute node. This uses a tiny
# inline PBS script we build here, submit, wait, and pull back.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cp ../lab01/heat2D .
/usr/bin/time -v ./heat2D --N 512 --steps 500 --snapEvery 100 --outDir ./out 2> timeReport.txt
cat timeReport.txt
'''
pbsHead = pbsHeader(name='lab02Timer', project=env['HPC_PROJECT'],
                    queue=env.get('HPC_QUEUE','debug'),
                    walltime='00:10:00', filesystems='home:eagle',
                    outPath=env['HPC_LAB_DIR']+'/timer.out')
pbsPath = labDir / 'timerJob.pbs'
pbsPath.write_text(pbsHead + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/timerJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/timerJob.pbs')
print('submitted:', jobID)


In [ ]:
# [Hub] Wait for it. Debug queue is usually 1-3 minutes.
waitJob(jobID, pollSeconds=15, maxSeconds=900)
sshGet(env['HPC_LAB_DIR']+'/timer.out',       str(labDir/'timer.out'))
sshGet(env['HPC_LAB_DIR']+'/timeReport.txt', str(labDir/'timeReport.txt'))
print('=== job stdout ===')
print((labDir/'timer.out').read_text()[:600])
print('=== /usr/bin/time -v report ===')
print((labDir/'timeReport.txt').read_text())


### 🖊️ Read the report

1. **`Elapsed (wall clock) time`** — the wall-time number, in `H:MM:SS.ss` format.
2. **`User time` + `System time`** — sum these; is the total close to wall (single-core-bound), well under wall (I/O-bound), or well over (already parallel)?
3. **`Maximum resident set size`** — peak memory usage. Useful when you scale N up.
4. **`Minor (reclaiming a frame) page faults`** — a proxy for memory pressure; if this is huge, your working set doesn't fit in RAM.
5. **`Percent of CPU this job got`** — 100% means one core saturated; more means multi-core; less means blocked on I/O.


In [ ]:
checkpoint("Part 1 - timers on Crux", [
    check("job finished", fileExists(str(labDir/'timer.out'))),
    check("/usr/bin/time report captured", fileExists(str(labDir/'timeReport.txt'))),
    check("report mentions Elapsed time",
          fileContains(str(labDir/'timeReport.txt'), 'Elapsed')),
])


## Part 2 · Draw a real roofline for Crux

labDD Part 5 walked through the roofline model with made-up numbers. Now you measure the two ceilings for real, on a Crux compute node.

- **Peak memory bandwidth** — via a [STREAM triad](https://www.cs.virginia.edu/stream/) kernel (`a[i] = b[i] + scalar * c[i]` over a huge array). This is *sustained* DRAM bandwidth, which is what real codes see; vendor peak numbers usually assume perfect cache reuse that no real code has.
- **Peak FLOP rate** — via a small in-cache FMA loop. Not exactly Linpack, but close enough to draw the compute ceiling.

Then you place lab 01's heat-stencil measurement on the same plot. If the stencil is anywhere near the memory-BW ceiling, congratulations — you're memory-bandwidth limited and further optimizations should target reuse, not FLOPs.


In [ ]:
# [Hub] Write the two microbenchmarks.
(labDir/'streamTriad.c').write_text('/* streamTriad.c - measure sustained memory bandwidth (STREAM triad).\n * a[i] = b[i] + scalar * c[i]  over a large array.\n * Reports GB/s and GFLOP/s for the roofline plot.\n * Build:  cc -O3 -Wall -o streamTriad streamTriad.c\n * Run:    ./streamTriad [--N 100000000] [--reps 10]\n */\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <time.h>\n\nstatic double wallSeconds(void) {\n    struct timespec ts; clock_gettime(CLOCK_MONOTONIC, &ts);\n    return ts.tv_sec + ts.tv_nsec * 1e-9;\n}\n\nint main(int argc, char **argv) {\n    long N = 100000000;      /* 100M doubles => 800MB per array; fits in most nodes */\n    int  reps = 10;\n    for (int i = 1; i < argc; ++i) {\n        if      (!strcmp(argv[i], "--N")    && i+1 < argc) N    = atol(argv[++i]);\n        else if (!strcmp(argv[i], "--reps") && i+1 < argc) reps = atoi(argv[++i]);\n    }\n\n    double *a = malloc(N * sizeof(double));\n    double *b = malloc(N * sizeof(double));\n    double *c = malloc(N * sizeof(double));\n    if (!a || !b || !c) { fprintf(stderr, "alloc failed\\n"); return 1; }\n\n    /* Touch each array once so pages are actually allocated (first-touch). */\n    for (long i = 0; i < N; ++i) { a[i] = 1.0; b[i] = 2.0; c[i] = 3.0; }\n\n    double scalar = 3.14;\n    double bestSec = 1e30;\n    for (int r = 0; r < reps; ++r) {\n        double t0 = wallSeconds();\n        for (long i = 0; i < N; ++i) a[i] = b[i] + scalar * c[i];\n        double dt = wallSeconds() - t0;\n        if (dt < bestSec) bestSec = dt;\n    }\n\n    /* Bytes moved per iter: read b, read c, write a  => 3 * 8 * N */\n    double bytesMoved = 3.0 * 8.0 * (double)N;\n    /* FLOPs per iter: one multiply + one add per element */\n    double flops      = 2.0 * (double)N;\n\n    double gbPerSec   = bytesMoved / bestSec / 1e9;\n    double gflopsRate = flops       / bestSec / 1e9;\n\n    printf("STREAM triad  N=%ld  bestSec=%.4f  BW=%.2f GB/s  FLOPs=%.2f GFLOP/s\\n",\n           N, bestSec, gbPerSec, gflopsRate);\n    free(a); free(b); free(c);\n    return 0;\n}\n')
(labDir/'peakFlop.c').write_text('/* peakFlop.c - approximate peak FMA throughput.\n * Independent chains of a[i] = a[i] * x + y so the compiler can vectorize\n * and the CPU can pipeline. Not a "true" LINPACK number, but close enough\n * to draw a compute ceiling on a roofline plot.\n * Build:  cc -O3 -Wall -march=native -ffast-math -o peakFlop peakFlop.c\n * Run:    ./peakFlop [--N 8388608] [--reps 200]\n */\n#include <stdio.h>\n#include <stdlib.h>\n#include <string.h>\n#include <time.h>\n\nstatic double wallSeconds(void) {\n    struct timespec ts; clock_gettime(CLOCK_MONOTONIC, &ts);\n    return ts.tv_sec + ts.tv_nsec * 1e-9;\n}\n\nint main(int argc, char **argv) {\n    long N = 8*1024*1024;    /* 8M doubles => 64MB; fits in most L3 caches */\n    int  reps = 200;\n    for (int i = 1; i < argc; ++i) {\n        if      (!strcmp(argv[i], "--N")    && i+1 < argc) N    = atol(argv[++i]);\n        else if (!strcmp(argv[i], "--reps") && i+1 < argc) reps = atoi(argv[++i]);\n    }\n\n    double *a = aligned_alloc(64, N * sizeof(double));\n    for (long i = 0; i < N; ++i) a[i] = 1.0;\n    double x = 1.0000001, y = 0.9999999;\n\n    /* Warm up */\n    for (int r = 0; r < 5; ++r)\n        for (long i = 0; i < N; ++i) a[i] = a[i] * x + y;\n\n    double bestSec = 1e30;\n    for (int r = 0; r < reps; ++r) {\n        double t0 = wallSeconds();\n        for (long i = 0; i < N; ++i) a[i] = a[i] * x + y;\n        double dt = wallSeconds() - t0;\n        if (dt < bestSec) bestSec = dt;\n    }\n\n    /* Each FMA counts as 2 FLOPs */\n    double flops = 2.0 * (double)N;\n    double gflopsRate = flops / bestSec / 1e9;\n    printf("Peak FMA test  N=%ld  bestSec=%.6f  peak~%.2f GFLOP/s\\n",\n           N, bestSec, gflopsRate);\n    /* Prevent DCE */\n    double sum = 0.0; for (long i = 0; i < N; ++i) sum += a[i];\n    if (sum == 0.0) printf("impossible\\n");\n    free(a);\n    return 0;\n}\n')
showFile(labDir/'streamTriad.c', language='c', maxLines=40, title='streamTriad.c')


In [ ]:
# [Hub -> Crux] Ship + build + run both microbenches on a compute node.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cc -O3 -Wall -o streamTriad streamTriad.c
cc -O3 -Wall -march=native -ffast-math -o peakFlop peakFlop.c
echo === streamTriad ===
./streamTriad --N 100000000 --reps 10
echo === peakFlop ===
./peakFlop --N 8388608 --reps 200
'''
pbsPath = labDir/'roofJob.pbs'
pbsPath.write_text(pbsHeader(name='lab02Roof', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             walltime='00:15:00', filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/roof.out') + jobBody)
sshPut(str(labDir/'streamTriad.c'), env['HPC_LAB_DIR']+'/streamTriad.c')
sshPut(str(labDir/'peakFlop.c'),    env['HPC_LAB_DIR']+'/peakFlop.c')
sshPut(str(pbsPath),                env['HPC_LAB_DIR']+'/roofJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/roofJob.pbs')
print('submitted:', jobID); waitJob(jobID, pollSeconds=15, maxSeconds=900)
sshGet(env['HPC_LAB_DIR']+'/roof.out', str(labDir/'roof.out'))
print((labDir/'roof.out').read_text())


In [ ]:
# [Hub] Parse the two numbers out of roof.out, plot the roofline, place the
# lab01 heat-stencil measurement on it.
import re
import pandas as pd, numpy as np, matplotlib.pyplot as plt
applyHouseStyle()
txt = (labDir/'roof.out').read_text()
bwMatch    = re.search(r'BW=([\d.]+) GB/s', txt)
peakMatch  = re.search(r'peak~([\d.]+) GFLOP/s', txt)
peakBW     = float(bwMatch.group(1))    if bwMatch    else 100.0
peakCompute= float(peakMatch.group(1))  if peakMatch  else 500.0
print(f'measured peak BW      = {peakBW:.1f} GB/s')
print(f'measured peak compute = {peakCompute:.1f} GFLOP/s')
print(f'ridge intensity       = {peakCompute/peakBW:.2f} FLOP/byte')

# lab01 heat-stencil measurement: MLUP/s -> GFLOP/s (5 FLOPs per lattice update -
# 4 adds + 1 multiply in the update expression; adjust if you counted differently).
# Arithmetic intensity: 5 FLOPs / (5 doubles * 8 B) ~ 0.125 FLOP/byte (best case).
lab01CSV = labDir.parent / 'lab01' / 'out' / 'timings.csv'
if lab01CSV.exists():
    l01 = pd.read_csv(lab01CSV)
    stencilGflops = l01['mlups'].iloc[0] * 5 / 1000.0    # (M lat-upd/s) * (5 flop/lat-upd) / 1000 -> GFLOP/s
    stencilI      = 0.125
    print(f'lab01 stencil: {stencilGflops:.2f} GFLOP/s at I~{stencilI} FLOP/byte')
else:
    stencilGflops = 0.10; stencilI = 0.125
    print('(lab01 CSV not found on Hub - using placeholder point)')

intensities = np.logspace(-2, 2, 200)
ceiling     = np.minimum(peakCompute, intensities * peakBW)
fig, ax = plt.subplots()
ax.plot(intensities, ceiling, color='black', lw=2, label='roofline')
ax.axhline(peakCompute, ls=':', alpha=0.5, label=f'peak compute {peakCompute:.0f}')
ax.plot(intensities, intensities*peakBW, ls=':', alpha=0.5, label=f'peak BW {peakBW:.0f} GB/s')
ax.plot([stencilI], [stencilGflops], marker='X', markersize=14,
        label=f'lab01 stencil ({stencilGflops:.1f} GFLOP/s)')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('arithmetic intensity (FLOP/byte)')
ax.set_ylabel('performance (GFLOP/s)')
ax.set_xlim(0.01, 100); ax.set_ylim(0.1, 2000)
ax.grid(True, which='both', alpha=0.3)
ax.legend(loc='lower right', fontsize=8)
ax.set_title('Crux roofline (measured)')
saveFigure(fig, 'rooflineCrux', figuresDir=str(labDir/'figures'))


In [ ]:
checkpoint("Part 2 - real roofline", [
    check("roof.out captured", fileExists(str(labDir/'roof.out'))),
    check("roofline figure written", fileExists(str(labDir/'figures'/'rooflineCrux.pdf'))),
])


## Part 3 · `perf stat` · what the CPU counters see

Wall time tells you *how long* your program took. Hardware counters tell you *what it was doing*. The Linux tool [`perf stat`](https://perf.wiki.kernel.org/index.php/Tutorial) reports:

| Counter | What it measures | What high/low means |
|---|---|---|
| **cycles** | CPU clocks elapsed | just the denominator for the ratios below |
| **instructions** | retired instructions | high = doing work; low = stalled |
| **IPC** (instructions/cycle) | pipeline utilization | 3+ = great; <1 = stalled on memory or branches |
| **cache-misses** | last-level cache misses | high = memory-bound |
| **cache-references** | LLC accesses attempted | denominator for miss rate |
| **branch-misses** / **branches** | branch mispred rate | high = unpredictable control flow |

For an HPC stencil, the tell-tale pattern is **low IPC and high cache-miss rate** — exactly the memory-bound story the roofline told you in Part 2.


In [ ]:
# [Hub -> Crux] Run heat2D under perf stat on a compute node.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
module load perf 2>/dev/null || true          # not always a module; often just on PATH
perf stat -e cycles,instructions,cache-misses,cache-references,branch-misses,branches \\
    ./heat2D --N 512 --steps 500 --snapEvery 100 --outDir ./out 2> perfReport.txt
cat perfReport.txt
'''
pbsPath = labDir/'perfJob.pbs'
pbsPath.write_text(pbsHeader(name='lab02Perf', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             walltime='00:10:00', filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/perf.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/perfJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/perfJob.pbs'); waitJob(jobID, 15, 900)
sshGet(env['HPC_LAB_DIR']+'/perf.out',       str(labDir/'perf.out'))
sshGet(env['HPC_LAB_DIR']+'/perfReport.txt', str(labDir/'perfReport.txt'))
print((labDir/'perfReport.txt').read_text())


### 🖊️ Compute your ratios

From the numbers above:

1. **IPC** = instructions / cycles. Above 2 is very good; below 1 means you're stalled most of the time.
2. **LLC miss rate** = cache-misses / cache-references. Above ~20% for an HPC kernel is a strong "memory-bound" signal.
3. **Branch miss rate** = branch-misses / branches. Below 1% is healthy; the heat stencil should be near zero because its inner loops are predictable.

Do these numbers match the roofline story in Part 2? A memory-bound kernel should show low IPC and high cache miss rate; a compute-bound kernel should show high IPC and low cache miss rate.


In [ ]:
checkpoint("Part 3 - perf stat", [
    check("perf report captured", fileExists(str(labDir/'perfReport.txt'))),
    check("perf reported instructions",
          fileContains(str(labDir/'perfReport.txt'), 'instructions')),
])


## Part 4 · Measure power · joules per timestep

HPC allocations are measured in **node-hours**, but the physical resource being consumed is *energy*. A kernel that gets the same answer in half the time at the same power draw is twice as efficient. A kernel that gets the same answer at twice the power draw in half the time is a wash.

On x86 nodes with RAPL support, `perf stat -e power/energy-pkg/` reports package energy in **joules** for the duration of the measured region. On Crux specifically, PBS also records job energy in its accounting log; both agree to within a few percent.

Extend `timings.csv` with an `energy_j` column: for the same run, how many joules did the whole job cost, and how many joules per timestep? A watt of average power for one second is one joule.


In [ ]:
# [Hub -> Crux] perf stat with the RAPL energy counter.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
perf stat -e power/energy-pkg/,power/energy-ram/ \\
    ./heat2D --N 512 --steps 500 --snapEvery 100 --outDir ./out 2> energyReport.txt
cat energyReport.txt
"""Fall back to PBS accounting if perf doesnt have access to RAPL on this node."""
echo === PBS accounting for this job ===
qstat -f -x $PBS_JOBID | grep -iE "resources_used.*(energy|power|watt)" || echo "(no PBS energy line - perf output above is authoritative)"
'''
pbsPath = labDir/'powerJob.pbs'
pbsPath.write_text(pbsHeader(name='lab02Power', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             walltime='00:10:00', filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/power.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/powerJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/powerJob.pbs'); waitJob(jobID, 15, 900)
sshGet(env['HPC_LAB_DIR']+'/power.out',        str(labDir/'power.out'))
sshGet(env['HPC_LAB_DIR']+'/energyReport.txt', str(labDir/'energyReport.txt'))
print((labDir/'energyReport.txt').read_text())
print('--- full stdout (PBS energy line, if any):')
print((labDir/'power.out').read_text()[-600:])


In [ ]:
# [Hub] Parse energy out and append an energy_j column to timings.csv.
import re, pandas as pd
txt = (labDir/'energyReport.txt').read_text()
joulesPkg = None
for line in txt.splitlines():
    m = re.search(r'([\d.,]+)\s+Joules\s+power/energy-pkg/', line)
    if m: joulesPkg = float(m.group(1).replace(',', ''))
if joulesPkg is None:
    print('perf did NOT report RAPL energy on this node.')
    print('Falling back to a placeholder; if PBS logged energy above, use that number.')
    joulesPkg = 0.0
else:
    print(f'package energy for this run: {joulesPkg:.1f} J')

# Append to lab02's local timings.csv (schema-compatible with lab01).
outCSV = labDir/'out'/'timings.csv'
outCSV.parent.mkdir(exist_ok=True)
newRow = {'lab':'02','variant':'perfstat','N':512,'steps':500,'threads':1,'ranks':1,
          'wall_s':None,'mlups':None,'io_s':None,'energy_j':joulesPkg}
if outCSV.exists():
    existing = pd.read_csv(outCSV)
    combined = pd.concat([existing, pd.DataFrame([newRow])], ignore_index=True)
else:
    combined = pd.DataFrame([newRow])
combined.to_csv(outCSV, index=False)
print(f'wrote row to {outCSV}:'); print(combined.tail())


In [ ]:
checkpoint("Part 4 - power", [
    check("energy report captured", fileExists(str(labDir/'energyReport.txt'))),
    check("timings.csv has an energy_j column",
          lambda: ('energy_j' in pd.read_csv(labDir/'out'/'timings.csv').columns,
                   'energy_j column present')),
])


## Part 5 · Controlled A/B experiment · does `-ffast-math` help this kernel?

labCC Part 4 warned that `-ffast-math` can change your answer. Here you find out whether it helps *this* kernel — and by how much — while checking that the conservation drift stays acceptable.

Discipline for any A/B experiment in this course:

1. **Change exactly one thing** — the compile flag, not the compiler, not the input, not the node.
2. **Measure both.** Not just "the fast one is faster" — you need the *baseline* number too, from the same node, same day.
3. **Check correctness.** If the new version gives a different answer, the speedup is meaningless. Compare against lab 01's science.csv conservation check.
4. **Plot both on the same axes.** Overlay the fast and safe points on the same roofline.


In [ ]:
# [Hub -> Crux] Build heat2D twice with different flag sets, run both,
# capture both wall + conservation drift.
jobBody = f'''cd {env["HPC_LAB_DIR"]}
cp ../lab01/heat2D.c .
echo === build safe ===
cc -O3 -Wall -o heatSafe heat2D.c -lm
echo === build fast ===
cc -O3 -march=native -ffast-math -Wall -o heatFast heat2D.c -lm
echo === run safe ===
./heatSafe --N 512 --steps 1000 --snapEvery 200 --outDir ./outSafe
echo === run fast ===
./heatFast --N 512 --steps 1000 --snapEvery 200 --outDir ./outFast
'''
pbsPath = labDir/'abJob.pbs'
pbsPath.write_text(pbsHeader(name='lab02AB', project=env['HPC_PROJECT'],
                             queue=env.get('HPC_QUEUE','debug'),
                             walltime='00:15:00', filesystems='home:eagle',
                             outPath=env['HPC_LAB_DIR']+'/ab.out') + jobBody)
sshPut(str(pbsPath), env['HPC_LAB_DIR']+'/abJob.pbs')
jobID = submitJob(env['HPC_LAB_DIR']+'/abJob.pbs'); waitJob(jobID, 15, 900)
sshGet(env['HPC_LAB_DIR']+'/ab.out', str(labDir/'ab.out'))
sshGet(env['HPC_LAB_DIR']+'/outSafe/timings.csv', str(labDir/'timingsSafe.csv'))
sshGet(env['HPC_LAB_DIR']+'/outFast/timings.csv', str(labDir/'timingsFast.csv'))
sshGet(env['HPC_LAB_DIR']+'/outSafe/science.csv', str(labDir/'scienceSafe.csv'))
sshGet(env['HPC_LAB_DIR']+'/outFast/science.csv', str(labDir/'scienceFast.csv'))
print((labDir/'ab.out').read_text()[-800:])


In [ ]:
# [Hub] Compare the two runs: wall, MLUP/s, sum-of-u drift.
import pandas as pd
safe = pd.read_csv(labDir/'timingsSafe.csv').iloc[-1]
fast = pd.read_csv(labDir/'timingsFast.csv').iloc[-1]
safeSci = pd.read_csv(labDir/'scienceSafe.csv')
fastSci = pd.read_csv(labDir/'scienceFast.csv')
safeDrift = abs(safeSci['sum_u'].iloc[-1] - safeSci['sum_u'].iloc[0]) / abs(safeSci['sum_u'].iloc[0])
fastDrift = abs(fastSci['sum_u'].iloc[-1] - fastSci['sum_u'].iloc[0]) / abs(fastSci['sum_u'].iloc[0])

comparison = pd.DataFrame({
    'variant':    ['-O3 (safe)', '-O3 -ffast-math'],
    'wall_s':     [safe['wall_s'], fast['wall_s']],
    'mlups':      [safe['mlups'],  fast['mlups']],
    'drift_rel':  [safeDrift,      fastDrift],
})
comparison['speedup'] = comparison.loc[0,'wall_s'] / comparison['wall_s']
print(comparison.to_string(index=False))

if fastDrift > 1e-8:
    showNote(f'-ffast-math conservation drift {fastDrift:.2e} exceeds 1e-8 threshold - '
             'the numeric answer changed materially. For a paper, cite BOTH numbers and '
             'flag the trade-off.', kind='warn')
else:
    showNote(f'-ffast-math kept conservation drift under 1e-8 for this kernel. Speedup: '
             f'{comparison.iloc[1]["speedup"]:.2f}x.', kind='ok')


In [ ]:
checkpoint("Part 5 - A/B experiment", [
    check("safe run produced timings", fileExists(str(labDir/'timingsSafe.csv'))),
    check("fast run produced timings", fileExists(str(labDir/'timingsFast.csv'))),
    check("conservation drift measured for both",
          lambda: (fileExists(str(labDir/'scienceSafe.csv'))()[0] and
                   fileExists(str(labDir/'scienceFast.csv'))()[0],
                   'both science.csv files present')),
])


## Part 6 · What comes next

You now have four ways to measure a Crux run and one way to compare two runs honestly. From lab 03 onward, every optimization you propose gets held to the same discipline:

| Measurement | Answers |
|---|---|
| Wall + `/usr/bin/time -v` | how long, how much memory, how CPU-utilized |
| Roofline placement | am I memory-bound or compute-bound? |
| `perf stat` counters | IPC, cache misses, branch prediction — the *why* |
| RAPL energy | joules per timestep — the sustainability number |
| A/B with the science check | did I speed up without changing the answer? |

### Lab 03 preview · OpenMP

Lab 03 adds `#pragma omp parallel for` above the outer stencil loop and re-runs today's measurements at 1, 2, 4, 8, 16, 32, 64 threads. Predictions from today:

- **The roofline predicts** the stencil can gain from more compute only until it hits the memory-BW slope; then adding threads does nothing.
- **The `perf stat` cache-miss rate** should stay similar per thread — the working set per thread shrinks, but the total DRAM traffic doesn't change.
- **The energy** should scale sub-linearly — more cores, but shorter wall time.

You'll test each prediction in lab 03, on the same node, with the same measurements.


## Wrap up

Serial baseline in lab 01 → real measurement in lab 02. Every parallelization from here is a bet the roofline lets you place before you make it, and the `timings.csv` schema lets you compare directly.


### Lab scorecard


In [ ]:
labSummary("Performance Measurement")


---
### One-minute feedback

What worked, what didn't, what should be clearer.


In [ ]:
feedback("Performance Measurement")
